In [1]:
import torch
import torch.nn as nn
import json
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [2]:
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.block(x) + x)


class ResNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, channels=32, num_blocks=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            *[ResBlock(channels) for _ in range(num_blocks)],
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, num_classes),
        )

    def forward(self, x):
        return self.net(x)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

epoch = 5
batch_size = 128
lr = 1e-3

train_loader = DataLoader( datasets.FashionMNIST("data", True, download=True, transform=transforms.ToTensor()), batch_size=batch_size, shuffle=True)

val_loader = DataLoader(datasets.FashionMNIST("data", False, download=True, transform=transforms.ToTensor()), batch_size=256)

model = ResNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.CrossEntropyLoss()

for e in range(epoch):

    model.train()
    loss_sum = correct = total = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = loss_fn(out, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        loss_sum += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    model.eval()
    val_loss = val_correct = val_total = 0

    for x, y in val_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = loss_fn(out, y)

        val_loss += loss.item() * x.size(0)
        val_correct += (out.argmax(1) == y).sum().item()
        val_total += y.size(0)

    print(
        e + 1,
        loss_sum / total,
        correct / total,
        val_loss / val_total,
        val_correct / val_total
    )

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 204kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.78MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 26.3MB/s]


1 0.7003170139312744 0.7689166666666667 0.6211378575325012 0.7634
2 0.38246300489107765 0.8662166666666666 0.4541657709121704 0.8361
3 0.319710657898585 0.8869333333333334 0.3623375861644745 0.8727
4 0.28845540320078533 0.8988666666666667 0.3701248893737793 0.8663
5 0.26377157746950786 0.9061833333333333 0.5057135084152221 0.8354
